# Математическое описание класса LinearRegression

В данном документе приведены полные математические формулы, реализованные в коде пользовательского класса линейной регрессии на базе библиотеки `numpy`.

---

## 1. Метод `predict(X)` — Предсказание модели

Модель вычисляет линейную комбинацию признаков с текущими весами и добавляет свободный коэффициент (сдвиг):

$$ \hat{y} = X \cdot w + b $$

* **$\hat{y}$** (в коде `y_pred`) — вектор предсказанных значений модели.
* **$X$** — матрица признаков размерности $(n \times m)$.
* **$w$** (в коде `self.weight`) — вектор весов размерности $(m \times 1)$.
* **$b$** (в коде `self.intercept`) — скалярное значение сдвига (intercept).

---

## 2. Метод `MSE(pred, y)` — Функция потерь (Elastic Net)

Вычисляется полная ошибка, которая складывается из среднеквадратичной ошибки (MSE), а также штрафов $L_1$ (Lasso) и $L_2$ (Ridge):

$$ \text{Loss} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}^{(i)} - y^{(i)})^2 + \lambda_1 \sum_{j=1}^{m} |w_j| + \lambda_2 \sum_{j=1}^{m} w_j^2 $$

* **$n$** — количество объектов в выборке (длина вектора $y$).
* **$y^{(i)}$** — истинное значение целевой переменной для $i$-го объекта.
* **$\hat{y}^{(i)}$** — предсказанное значение модели для $i$-го объекта.
* **$\lambda_1$** (в коде `self.l1`) — коэффициент L1-регуляризации.
* **$\lambda_2$** (в коде `self.l2`) — коэффициент L2-регуляризации.
* **$w_j$** — значение $j$-го веса модели.

---

## 3. Метод `fit(X, y)` — Итеративный градиентный спуск

### Расчет градиентов (частных производных)
Градиенты вычисляются как сумма производных от базовой функции MSE и соответствующих штрафов регуляризации.

* **Градиент по весам ($\nabla_w \text{Loss}$ / в коде `gr_weight`):**
  $$ \nabla_w \text{Loss} = \frac{2}{n} X^T (\hat{y} - y) + \lambda_1 \cdot \text{sign}(w) + 2 \cdot \lambda_2 \cdot w $$

* **Градиент по сдвигу ($\nabla_b \text{Loss}$ / в коде `gr_intercept`):**
  $$ \nabla_b \text{Loss} = \frac{2}{n} \sum_{i=1}^{n} (\hat{y}^{(i)} - y^{(i)}) $$

### Обновление параметров (градиентный шаг)
Параметры модели сдвигаются в сторону, противоположную вектору градиента, с учетом шага обучения:

$$ w^{(\text{new})} = w^{(\text{old})} - \eta \cdot \nabla_w \text{Loss} $$

$$ b^{(\text{new})} = b^{(\text{old})} - \eta \cdot \nabla_b \text{Loss} $$

* **$\eta$** (в коде `self.learning_rate`) — скорость обучения (learning rate).

### Критерий ранней остановки
Итерационный цикл прерывается досрочно, если абсолютное изменение функции потерь между текущей и предыдущей эпохами падает ниже заданного порога:

$$ |\text{Loss}^{(\text{current})} - \text{Loss}^{(\text{previous})}| < \epsilon $$

* **$\epsilon$** (в коде `self.stop_gr`) — порог сходимости алгоритма.

---

## 4. Метод `fit_analytical(X, y)` — Точное нормальное уравнение

Если $L_1$-регуляризация отсутствует (`self.l1 == 0`), модель находит глобальный минимум функции потерь за один шаг аналитическим путем с помощью модифицированного уравнения Ridge-регрессии:

$$ \omega = (X_{\text{biased}}^T X_{\text{biased}} + \lambda_2 I^*)^{-1} X_{\text{biased}}^T y $$

* **$X_{\text{biased}}$** — расширенная матрица признаков, в которую первым столбцом добавлены единицы $\mathbf{1}$ для автоматического вычисления свободного коэффициента $b$:
  $$ X_{\text{biased}} = [\mathbf{1} \mid X] $$
* **$\omega$** (в коде `params`) — объединенный вектор искомых параметров, где первый элемент — это сдвиг, а последующие — веса признаков:
  $$ \omega = [b, w_1, w_2, \dots, w_m]^T $$
* **$I^*$** — модифицированная единичная матрица размера $((m+1) \times (m+1))$. Её самый первый элемент (строка 0, столбец 0) занулен, чтобы регуляризация **не накладывала штраф** на свободный коэффициент $b$:
  $$ I^* = \begin{pmatrix} 0 & 0 & \dots & 0 \\ 0 & 1 & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \dots & 1 \end{pmatrix} $$
* **$^{-1}$** — операция нахождения обратной (псевдообратной Мура-Пенроуза) матрицы, реализованная через стабильную функцию `np.linalg.pinv`.


### Аналитический метод через матрицы

$$\omega = (X^T X)^{-1} X^T y$$


### Аналитический метод с Ridge регуляризацией

$$\omega = (X^T X + \lambda I)^{-1} X^T y$$


* **$\omega$** — итоговый вектор весов (включая сдвиг, если к $X$ добавлен единичный столбец)
* **$X$** — матрица признаков
* **$X^T$** — транспонированная матрица признаков
* **$\lambda$** — коэффициент регуляризации (в коде это параметр `l2`)
* **$I$** — единичная матрица соответствующего размера с единицами по главной диагонали и нулями в остальных ячейках
* **$^{-1}$** — операция нахождения обратной матрицы
* **$y$** — вектор целевых значений.

In [56]:
import numpy as np

class LinearRegression:
    """
    Модель строит линейную комбинацию входных признаков и предсказывает
    непрерывное целевое значение. Обучение выполняется методом градиентного
    спуска с поддержкой критерия ранней остановки и гибкой регуляризации
    ElasticNet (комбинация L1 и L2 штрафов).

    Аргументы инициализации:
    ------------------------
    max_iter : int, default=1500
        Максимальное количество эпох градиентного спуска для обучения модели.
    learning_rate : float, default=0.01
        Скорость обучения, контролирующая величину обновления весов.
    stop_gr : float, default=1e-4
        Критерий ранней остановки. Обучение прерывается, если абсолютное изменение
        функции потерь MSE между соседними эпохами становится меньше этого значения.
    l1, l2 : float, default=0.0
        Коэффициенты L1-регуляризации (Lasso) и L2-регуляризации (Ridge). Отвечают за отбор признаков 
        путем зануления неважных весов. При l1=0.0 и l2>0.0 модель активирует Ridge (L2), при l1>0.0 
        и l2=0.0 активируется Lasso (L1). При одновременном l1>0.0 и l2>0.0 модель работает в режиме ElasticNet.

    Основные методы:
    ----------------
    MSE(pred, y)
        Вычисляет среднеквадратичную ошибку с учетом добавленных штрафов L1 и L2 регуляризаций.
    fit(X, y)
        Запускает цикл градиентного спуска для настройки весов и сдвига на обучающей выборке.
    fit_analytical(X, y)
        Находит оптимальные веса и сдвиг аналитическим путем через нормальное уравнение (с поддержкой L2-штрафа).
    predict(X)
        Предсказывает финальные непрерывные значения на основе рассчитанных весов и сдвига.

    Внутренние атрибуты:
    --------------------
    self.weight : numpy.ndarray
        Вектор подобранных весов модели. Длина вектора равна количеству признаков в матрице X.
    self.intercept : float
        Свободный коэффициент модели (сдвиг/bias).
    self.epoch : int
        Фактическое количество эпох, которое потребовалось модели до завершения обучения.
    """

    def __init__(self, max_iter = 1500, learning_rate = 0.01, l1 = 0, l2 = 0, stop_gr = 1e-4, auto_l2 = True):
        self.max_iter = max_iter
        self.learning_rate = learning_rate
        self.l1 = l1
        self.l2 = l2
        self.stop_gr = stop_gr
        self.auto_l2 = auto_l2
        

    def MSE(self,pred,y):
        loss = np.mean((pred - y)**2)
        l1_loss = self.l1 * np.sum(np.abs(self.weight)) 
        l2_loss = (self.l2 * np.sum(self.weight ** 2)) 
        return loss + l2_loss + l1_loss
    
    def fit(self,X,y):
        y = y.ravel()
        self.intercept = 0
        self.weight = np.zeros(X.shape[1])
        n = X.shape[0]

        prev_loss = float('inf')

        if self.l1 == 0 and self.auto_l2:
            self.epoch = None
            return self.fit_analytical(X,y)

        for epoch in range(self.max_iter):
            y_pred = X @ self.weight + self.intercept
            dif = y_pred - y

            loss = self.MSE(y_pred,y)
            if abs(prev_loss - loss) < self.stop_gr:
                break
            prev_loss = loss
            
            gr_weight = (2/n) * (X.T @ dif) + (self.l1 * np.sign(self.weight)) + ((2 * self.l2 * self.weight))
            gr_intersept = (2/n) * np.sum(dif)

            self.weight -= self.learning_rate * gr_weight
            self.intercept -= self.learning_rate * gr_intersept

        self.epoch = epoch
        return self

    def fit_analytical(self, X,y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).ravel()
        
        ones = np.ones((X.shape[0], 1)) 
        X_biased = np.hstack((ones, X))

        I = np.eye(X_biased.shape[1])
        I[0, 0] = 0  # Для отступа
        params = np.linalg.pinv(X_biased.T @ X_biased + self.l2 * I) @ X_biased.T @ y
        
        self.intercept = params[0]
        self.weight = params[1:]
        return self

    def predict(self,X):
        X = np.asarray(X, dtype=np.float64)
        return X @ self.weight + self.intercept

In [57]:
from sklearn.datasets import make_regression
from sklearn.linear_model import Lasso, Ridge

X, y = make_regression(n_samples=100, n_features=3, noise=10, random_state=42)

print("=== ТЕСТ 1: Чистая L1 Регуляризация (Сравнение с Lasso) ===")
alpha_l1 = 0.1

# Наша модель
custom_lasso = LinearRegression(l1=alpha_l1, l2=0).fit(X, y)
# Модель sklearn
sklearn_lasso = Lasso(alpha=alpha_l1, max_iter=10000).fit(X, y)

print(f"Веса (Custom):  {custom_lasso.weight}")
print(f"Веса (Sklearn): {sklearn_lasso.coef_}")
print(f"Сдвиг (Custom):  {custom_lasso.intercept:.4f}")
print(f"Сдвиг (Sklearn): {sklearn_lasso.intercept_:.4f}\n")


print("=== ТЕСТ 2: Чистая L2 Регуляризация (Сравнение с Ridge) ===")
alpha_l2 = 5.0

# Наша модель
custom_ridge = LinearRegression(l1=0, l2=alpha_l2, stop_gr=0).fit(X, y)
# Модель sklearn
sklearn_ridge = Ridge(alpha=alpha_l2).fit(X, y)

print(custom_ridge.epoch)

print(f"Веса (Custom):  {custom_ridge.weight}")
print(f"Веса (Sklearn): {sklearn_ridge.coef_}")
print(f"Сдвиг (Custom):  {custom_ridge.intercept:.4f}")
print(f"Сдвиг (Sklearn): {sklearn_ridge.intercept_:.4f}")

=== ТЕСТ 1: Чистая L1 Регуляризация (Сравнение с Lasso) ===
Веса (Custom):  [28.15708359 73.91741295 18.71795177]
Веса (Sklearn): [28.14318845 73.8804421  18.67019487]
Сдвиг (Custom):  1.2615
Сдвиг (Sklearn): 1.2501

=== ТЕСТ 2: Чистая L2 Регуляризация (Сравнение с Ridge) ===
None
Веса (Custom):  [25.7849356  70.72046023 17.32181568]
Веса (Sklearn): [25.7849356  70.72046023 17.32181568]
Сдвиг (Custom):  1.4558
Сдвиг (Sklearn): 1.4558
